# MT Quality Lab (EN→DE)
1. Translate + score FLORES-200 with NLLB-200-600M and opus-mt-en-de (chrF, BLEU, COMET)
2. Rule-based filter for a noisy parallel corpus
3. Manual audit of the filter (human step)
4. Manual error analysis (human step)

Set `SMOKE_TEST = True` for a quick CPU bug check; `False` for the real Colab T4 run.
Every number reported comes from `results.json`.

In [31]:
!pip install sacrebleu unbabel-comet
!pip install fasttext-wheel langid
!pip install datasets transformers evaluate

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [32]:
# ---- Top-level switches and config (edit here only) ----
SMOKE_TEST = False          # True: 20 FLORES sentences + 500 corpus pairs. False: full run.
SEED = 42

CONFIG = {
    # Part 1
    "flores_url": "https://dl.fbaipublicfiles.com/nllb/flores200_dataset.tar.gz",
    "systems": {
        "NLLB-200-600M": "facebook/nllb-200-distilled-600M",
        "opus-mt-en-de": "Helsinki-NLP/opus-mt-en-de",
    },
    "comet_model": "Unbabel/eamt22-cometinho-da",
    "num_beams": 4,
    "flores_n": 20 if SMOKE_TEST else None,         # None = all 1012
    # Part 2: corpus + filter thresholds
    "corpus": {"hf_id": "sentence-transformers/parallel-sentences-wikimatrix", "config": "en-de", "split": "train"},
    "corpus_n": 500 if SMOKE_TEST else 20000,        # seeded random sample of the corpus
    "filter": {
        "min_chars": 5,            # min characters per side
        "max_words": 100,          # max words per side
        "max_len_ratio": 2.0,      # max(len_src, len_tgt) / min(...), in characters
        "lid_min_conf": 0.5,       # min fastText confidence for the expected language
        "lid_url": "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.ftz",
    },
}

In [33]:
# ---- Setup: Colab detection, installs, Drive, output dir, seeds, device ----
import sys, os, json, subprocess, random, time, gc
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
PINS = ["transformers==4.57.6", "sacrebleu==2.6.0", "unbabel-comet==2.2.7", "datasets==5.0.1",
        "fasttext-wheel==0.9.2", "langid==1.1.6", "sentencepiece", "pandas", "setuptools<81"]
if IN_COLAB:
    # Installing unbabel-comet may change numpy; if Colab asks to restart the runtime, do so and rerun.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/mt_quality_lab")
else:
    BASE = Path("outputs")
# smoke and full runs never share caches/outputs
OUT = BASE / ("smoke" if SMOKE_TEST else "full")
OUT.mkdir(parents=True, exist_ok=True)
print("Colab:", IN_COLAB, "| outputs ->", OUT.resolve())

import numpy as np, pandas as pd, torch, transformers, sacrebleu, comet
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# device: CUDA -> MPS -> CPU
if torch.cuda.is_available(): DEVICE = "cuda"
elif torch.backends.mps.is_available(): DEVICE = "mps"
else: DEVICE = "cpu"
print("device:", DEVICE)

# results.json: merged, never overwritten wholesale
RESULTS_PATH = OUT / "results.json"
def load_results():
    return json.loads(RESULTS_PATH.read_text()) if RESULTS_PATH.exists() else {}
def save_results(key, value):
    r = load_results(); r[key] = value
    RESULTS_PATH.write_text(json.dumps(r, indent=2, ensure_ascii=False))

save_results("meta", {
    "smoke_test": SMOKE_TEST, "seed": SEED, "device": DEVICE, "on_colab": IN_COLAB,
    "python": sys.version.split()[0],
    "versions": {"torch": torch.__version__, "transformers": transformers.__version__,
                 "sacrebleu": sacrebleu.__version__, "unbabel-comet": comet.__version__ if hasattr(comet, "__version__") else "see pip",
                 "numpy": np.__version__, "pandas": pd.__version__},
    "model_ids": {**CONFIG["systems"], "comet": CONFIG["comet_model"]},
    "config": CONFIG,
})

Colab: False | outputs -> /home/jovyan/outputs/full
device: cuda


## Part 1: Translate and score (FLORES-200 devtest, eng_Latn → deu_Latn)

In [34]:
# ---- Load FLORES-200 devtest from Meta's official tarball (HF copies are gated/script-based) ----
import tarfile, urllib.request
flores_dir = BASE / "flores200_dataset"            # shared by smoke/full runs (raw data only)
src_f, ref_f = flores_dir / "devtest/eng_Latn.devtest", flores_dir / "devtest/deu_Latn.devtest"
if not (src_f.exists() and ref_f.exists()):
    tgz = BASE / "flores200_dataset.tar.gz"
    urllib.request.urlretrieve(CONFIG["flores_url"], tgz)
    with tarfile.open(tgz) as t: t.extractall(BASE)
src = src_f.read_text(encoding="utf-8").splitlines()
ref = ref_f.read_text(encoding="utf-8").splitlines()
assert len(src) == len(ref) == 1012
n = CONFIG["flores_n"] or len(src)
src, ref = src[:n], ref[:n]
print(f"FLORES devtest: {len(src)} sentences. Example:\n  EN: {src[0]}\n  DE: {ref[0]}")

FLORES devtest: 1012 sentences. Example:
  EN: "We now have 4-month-old mice that are non-diabetic that used to be diabetic," he added.
  DE: „Wir haben jetzt 4 Monate alte Mäuse, die Diabetes hatten und jetzt keinen mehr haben“, fügte er hinzu.


In [35]:
# ---- Translation (batched, beam 4, fp16 on CUDA, MPS->CPU fallback) ----
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

def translate(model_id, sentences, batch_size):
    tok = AutoTokenizer.from_pretrained(model_id, **({"src_lang": "eng_Latn"} if "nllb" in model_id else {}))
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id).eval()
    gen_kw = {"forced_bos_token_id": tok.convert_tokens_to_ids("deu_Latn")} if "nllb" in model_id else {}
    dev = DEVICE
    model.to(dev)
    if dev == "cuda": model.half()
    order = sorted(range(len(sentences)), key=lambda i: -len(sentences[i]))   # length-sorted batches
    hyps = [None] * len(sentences)
    for s in range(0, len(order), batch_size):
        idx = order[s:s + batch_size]
        batch = tok([sentences[i] for i in idx], return_tensors="pt", padding=True, truncation=True, max_length=256)
        try:
            with torch.no_grad():
                out = model.generate(**batch.to(dev), num_beams=CONFIG["num_beams"], max_new_tokens=256, **gen_kw)
        except RuntimeError as e:
            if dev != "mps": raise
            print("MPS error, falling back to CPU:", str(e)[:100]); dev = "cpu"; model.to("cpu")
            with torch.no_grad():
                out = model.generate(**batch.to("cpu"), num_beams=CONFIG["num_beams"], max_new_tokens=256, **gen_kw)
        for i, h in zip(idx, tok.batch_decode(out, skip_special_tokens=True)): hyps[i] = h
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return hyps

BATCH = 32 if DEVICE == "cuda" else 8
translations = {}
for name, model_id in CONFIG["systems"].items():
    path = OUT / f"translations_{name}.jsonl"
    if path.exists():                                   # cache: skip finished work
        rows = [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines()]
        if len(rows) == len(src): translations[name] = [r["hyp"] for r in rows]; print(name, "loaded from cache"); continue
    t0 = time.time()
    hyps = translate(model_id, src, BATCH)
    with open(path, "w", encoding="utf-8") as f:
        for i, (s, h, r) in enumerate(zip(src, hyps, ref)):
            f.write(json.dumps({"id": i, "src": s, "hyp": h, "ref": r}, ensure_ascii=False) + "\n")
    translations[name] = hyps
    print(f"{name}: {len(hyps)} sentences in {time.time()-t0:.0f}s")

NLLB-200-600M loaded from cache
opus-mt-en-de loaded from cache


In [36]:
# ---- Scoring: sacreBLEU chrF + BLEU (default settings) and Cometinho COMET ----
from sacrebleu.metrics import CHRF, BLEU
from comet import download_model, load_from_checkpoint

chrf_m, bleu_m = CHRF(), BLEU()
comet_model = load_from_checkpoint(download_model(CONFIG["comet_model"]))
COMET_GPUS = 1 if DEVICE == "cuda" else 0               # COMET: CUDA or CPU only (never MPS)

part1 = {"n_sentences": len(src), "comet_model": CONFIG["comet_model"], "systems": {}}
for name, hyps in translations.items():
    chrf, bleu = chrf_m.corpus_score(hyps, [ref]), bleu_m.corpus_score(hyps, [ref])
    data = [{"src": s, "mt": h, "ref": r} for s, h, r in zip(src, hyps, ref)]
    c = comet_model.predict(data, batch_size=64 if COMET_GPUS else 8, gpus=COMET_GPUS, progress_bar=False, num_workers=2)
    part1["systems"][name] = {
        "model_id": CONFIG["systems"][name],
        "chrF": chrf.score, "chrF_signature": str(chrf_m.get_signature()),
        "BLEU": bleu.score, "BLEU_signature": str(bleu_m.get_signature()),
        "COMET": float(c.system_score),
    }
save_results("part1", part1)

table = pd.DataFrame([{"system": k, "chrF": v["chrF"], "BLEU": v["BLEU"], "COMET": v["COMET"]}
                      for k, v in part1["systems"].items()]).round(3)
print(f"FLORES-200 devtest EN->DE, n={len(src)}" + ("  [SMOKE TEST - not a real result]" if SMOKE_TEST else ""))
print(table.to_string(index=False))
print("chrF sig:", next(iter(part1["systems"].values()))["chrF_signature"])
print("BLEU sig:", next(iter(part1["systems"].values()))["BLEU_signature"])

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

/home/jovyan/.local/lib/python3.11/site-packages/lightning_fabric/utilities/cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
Lightning automatically upgraded your loaded checkpo

FLORES-200 devtest EN->DE, n=1012
       system  chrF   BLEU  COMET
NLLB-200-600M 62.02 34.684  0.551
opus-mt-en-de 63.66 36.171  0.553
chrF sig: nrefs:1|case:mixed|eff:yes|nc:6|nw:0|space:no|version:2.6.0
BLEU sig: nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0


## Part 2: Corpus filter (WikiMatrix EN-DE, seeded random sample)

In [37]:
# ---- Load a seeded random sample of the noisy corpus ----
from datasets import load_dataset
C = CONFIG["corpus"]
ds = load_dataset(C["hf_id"], C["config"], split=C["split"])
total_rows = len(ds)
ds = ds.shuffle(seed=SEED).select(range(CONFIG["corpus_n"]))
corpus = pd.DataFrame({"row": range(len(ds)), "src": ds["english"], "tgt": ds["non_english"]})
print(f"{C['hf_id']} [{C['config']}/{C['split']}]: sampled {len(corpus)} of {total_rows} pairs (shuffle seed={SEED})")
corpus.head(3)

sentence-transformers/parallel-sentences-wikimatrix [en-de/train]: sampled 20000 of 344476 pairs (shuffle seed=42)


,row,src,tgt
0,0,The castle was originally known as Sarnen Cast...,Das Schloss war ursprünglich als Sarnenburg od...
1,1,The house was called Greenfields for many years.,Viele Jahre lang hieß das Landhaus Greenfields.
2,2,"The principal cities are Tampa, St. Petersburg...",Die größten Städte des Verdichtungsgebietes si...


In [38]:
# ---- Language ID: fastText lid.176 (fallback: langid) ----
try:
    import fasttext
    lid_path = BASE / "lid.176.ftz"
    if not lid_path.exists(): urllib.request.urlretrieve(CONFIG["filter"]["lid_url"], lid_path)
    _ft = fasttext.load_model(str(lid_path))
    LID_BACKEND = "fasttext lid.176.ftz"
    def lid(texts):
        labels, probs = _ft.predict([t.replace("\n", " ") for t in texts], k=1)
        return [(l[0].replace("__label__", ""), float(p[0])) for l, p in zip(labels, probs)]
except Exception as e:
    print("fastText unavailable, using langid:", e)
    import langid
    langid.set_languages(["en", "de"]); LID_BACKEND = "langid"
    def lid(texts):
        out = []
        for t in texts:
            l, s = langid.classify(t); out.append((l, 1.0 if s > 0 else 0.0))   # langid score is unnormalized
        return out
print("LID backend:", LID_BACKEND)

LID backend: fasttext lid.176.ftz


In [39]:
# ---- Filter rules. Every pair is tested against ALL rules; all failed rules are logged ----
import re, unicodedata
F = CONFIG["filter"]

def norm(s): return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", s)).strip().lower()
def digit_set(s):
    s = re.sub(r"(?<=\d)[.,\u00a0\u202f ](?=\d{3}\b)", "", s)    # 1,000 / 1.000 -> 1000
    return frozenset(re.findall(r"\d+", s))

RULES = ["empty", "duplicate", "length", "lang_id", "number_mismatch"]    # application order
df = corpus.copy()
s, t = df["src"].fillna(""), df["tgt"].fillna("")
fail = pd.DataFrame(index=df.index)

fail["empty"] = (s.str.strip() == "") | (t.str.strip() == "")
key = (s.map(norm) + "\t" + t.map(norm))
fail["duplicate"] = key.duplicated(keep="first")                          # later copies are removed
ls, lt = s.str.len(), t.str.len()
ratio = np.maximum(ls, lt) / np.maximum(np.minimum(ls, lt), 1)
fail["length"] = ((ls < F["min_chars"]) | (lt < F["min_chars"]) | (ratio > F["max_len_ratio"])
                  | (s.str.split().str.len() > F["max_words"]) | (t.str.split().str.len() > F["max_words"]))
ls_lab, lt_lab = lid(s.tolist()), lid(t.tolist())
fail["lang_id"] = [not (a[0] == "en" and a[1] >= F["lid_min_conf"] and b[0] == "de" and b[1] >= F["lid_min_conf"])
                   for a, b in zip(ls_lab, lt_lab)]
fail["number_mismatch"] = [digit_set(a) != digit_set(b) for a, b in zip(s, t)]

df["reason"] = fail.apply(lambda r: ";".join(k for k in RULES if r[k]), axis=1)
removed_mask = df["reason"] != ""

In [40]:
# ---- Report, save kept/removed, record in results.json ----
n_total, n_removed = len(df), int(removed_mask.sum())
first_fail = df.loc[removed_mask, "reason"].str.split(";").str[0].value_counts()   # sequential attribution
per_rule = {r: {"failed_any": int(fail[r].sum()),                                   # pairs failing this rule (overlaps possible)
                "removed_first": int(first_fail.get(r, 0))}                         # pairs removed when rules apply in order
            for r in RULES}
part2 = {"source": {**CONFIG["corpus"], "total_rows": total_rows, "sample_n": n_total,
                    "sampling": f"datasets shuffle(seed={SEED}).select(range({n_total}))"},
         "lid_backend": LID_BACKEND, "thresholds": F,
         "n_total": n_total, "n_removed": n_removed, "n_kept": n_total - n_removed,
         "pct_removed": 100 * n_removed / n_total, "per_rule": per_rule}
save_results("part2", part2)

clean = lambda d: d.assign(src=d["src"].str.replace(r"[\t\r\n]+", " ", regex=True), tgt=d["tgt"].str.replace(r"[\t\r\n]+", " ", regex=True))
clean(df[~removed_mask][["row", "src", "tgt"]]).to_csv(OUT / "kept.tsv", sep="\t", index=False)
clean(df[removed_mask][["row", "src", "tgt", "reason"]]).to_csv(OUT / "removed.tsv", sep="\t", index=False)

print(f"{'[SMOKE] ' if SMOKE_TEST else ''}removed {n_removed}/{n_total} ({part2['pct_removed']:.1f}%), kept {part2['n_kept']}")
print(pd.DataFrame(per_rule).T.to_string())
print("rules failed per removed pair:", df.loc[removed_mask, "reason"].str.count(";").add(1).value_counts().sort_index().to_dict())
df[removed_mask][["src", "tgt", "reason"]].sample(min(6, n_removed), random_state=SEED).to_string(max_colwidth=60)

removed 1669/20000 (8.3%), kept 18331
                 failed_any  removed_first
empty                     0              0
duplicate                 0              0
length                   21             21
lang_id                 987            981
number_mismatch         729            667
rules failed per removed pair: {1: 1601, 2: 68}


'                                                               src                                                          tgt           reason\n11267          Benth. & Hook., and Haloxylon recurvum sensu Bunge.           Benth. & Hook. und Haloxylon recurvum sensu Bunge.          lang_id\n11880  In the 1950s, Bober met François Truffaut and became his...  In den 1950er Jahren traf Bober François Truffaut und wu...  number_mismatch\n5027   AMNH FR 7434 (Early Eocene) Accipitridae gen. et sp. indet.  AMNH FR 7434 (Unteres Eozän) Accipitridae gen. et sp. in...          lang_id\n8898   This meant, for example, that from 7:00 pm all trains ha...  So sind beispielsweise ab 19 Uhr alle Züge mit Zugbeglei...  number_mismatch\n19419                             Earnhardt: Okay, just wondering.                                              In: Just Asked.           length\n1796   A character data book titled Nurarihyon no Mago: Officia...  Ein Sonderband mit dem Titel Nurarihyon no Mago: Offici

## Part 3: Manual audit (HUMAN STEP)

In [41]:
# ---- Export the blinded audit sheet (100 removed + 100 kept, shuffled, fixed seed) ----
# Never overwrites an existing sheet, so labels you already entered are safe.
sheet_path, key_path = OUT / "audit_sheet.csv", OUT / "audit_key.csv"
if sheet_path.exists():
    print("audit_sheet.csv already exists - not overwritten.")
else:
    rem = df[removed_mask].sample(min(100, int(removed_mask.sum())), random_state=SEED).assign(decision="removed")
    kep = df[~removed_mask].sample(min(100, int((~removed_mask).sum())), random_state=SEED).assign(decision="kept")
    aud = pd.concat([rem, kep]).sample(frac=1, random_state=SEED).reset_index(drop=True)
    aud["audit_id"] = range(len(aud))
    aud[["audit_id", "src", "tgt"]].assign(is_good_pair="", notes="").to_csv(sheet_path, index=False)
    aud[["audit_id", "row", "decision", "reason"]].to_csv(key_path, index=False)
    print(f"Exported {len(aud)} pairs ({len(rem)} removed, {len(kep)} kept) -> {sheet_path}")
    if SMOKE_TEST: print("[SMOKE] fewer than 100 removed pairs available, so the sheet is smaller than the real one.")
print("NEXT: fill is_good_pair (1 = correct translation pair, 0 = bad pair) and notes, save as audit_sheet_labeled.csv in", OUT)

audit_sheet.csv already exists - not overwritten.
NEXT: fill is_good_pair (1 = correct translation pair, 0 = bad pair) and notes, save as audit_sheet_labeled.csv in outputs/full


In [42]:
# ---- Audit analysis: run after you upload audit_sheet_labeled.csv ----
def wilson(k, n, z=1.96):
    if n == 0: return (float("nan"), float("nan"))
    p = k / n; d = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / d; h = z * ((p * (1 - p) / n + z**2 / (4 * n**2)) ** 0.5) / d
    return (100 * (c - h), 100 * (c + h))

lab_path = OUT / "audit_sheet_labeled.csv"
if not lab_path.exists():
    print("NOT RUN: audit_sheet_labeled.csv not found in", OUT)
else:
    lab = pd.read_csv(lab_path).merge(pd.read_csv(key_path), on="audit_id")
    assert lab["is_good_pair"].isin([0, 1]).all(), "every row needs is_good_pair = 0 or 1"
    out = {}
    for name, dec in [("removal_precision", "removed"), ("residual_noise_kept", "kept")]:
        d = lab[lab["decision"] == dec]; k, n = int((d["is_good_pair"] == 0).sum()), len(d)   # k = truly bad pairs
        lo, hi = wilson(k, n)
        out[name] = {"bad": k, "n": n, "pct": 100 * k / n, "wilson95_low": lo, "wilson95_high": hi}
        print(f"{name}: {k}/{n} = {100*k/n:.1f}%  (Wilson 95% CI {lo:.1f}-{hi:.1f}%)")
    out["note"] = "n=100 per group gives wide confidence intervals; treat as rough estimates."
    save_results("part3", out)
    print("Caution: n=100 per group gives wide uncertainty (see CI widths).")

removal_precision: 44/100 = 44.0%  (Wilson 95% CI 34.7-53.8%)
residual_noise_kept: 4/100 = 4.0%  (Wilson 95% CI 1.6-9.8%)
Caution: n=100 per group gives wide uncertainty (see CI widths).


## Part 4: Error analysis (HUMAN STEP)

In [43]:
# ---- Export error sheet: 30 random sentences from the best and worst system by COMET ----
CATEGORIES = ["terminology", "named entity", "number/date", "fluency", "omission/addition", "mistranslation", "none"]
sys_comet = {k: v["COMET"] for k, v in load_results()["part1"]["systems"].items()}
best, worst = max(sys_comet, key=sys_comet.get), min(sys_comet, key=sys_comet.get)
print("best:", best, "| worst:", worst, "| COMET:", {k: round(v, 3) for k, v in sys_comet.items()})

err_path = OUT / "error_sheet.csv"
if err_path.exists():
    print("error_sheet.csv already exists - not overwritten.")
else:
    parts = []
    for name in [best, worst]:
        rows = pd.DataFrame([json.loads(l) for l in (OUT / f"translations_{name}.jsonl").read_text(encoding="utf-8").splitlines()])
        parts.append(rows.sample(min(30, len(rows)), random_state=SEED).assign(system=name))
    es = pd.concat(parts)[["id", "system", "src", "ref", "hyp"]].assign(error_category="", notes="")
    es.to_csv(err_path, index=False)
    print(f"Exported {len(es)} rows -> {err_path}")
    if SMOKE_TEST: print("[SMOKE] only 20 sentences exist, so fewer than 30 per system.")
print("NEXT: fill error_category with one of", CATEGORIES, "and save as error_sheet_labeled.csv in", OUT)

best: opus-mt-en-de | worst: NLLB-200-600M | COMET: {'NLLB-200-600M': 0.551, 'opus-mt-en-de': 0.553}
error_sheet.csv already exists - not overwritten.
NEXT: fill error_category with one of ['terminology', 'named entity', 'number/date', 'fluency', 'omission/addition', 'mistranslation', 'none'] and save as error_sheet_labeled.csv in outputs/full


In [44]:
# ---- Error analysis: run after you upload error_sheet_labeled.csv ----
lab_path = OUT / "error_sheet_labeled.csv"
if not lab_path.exists():
    print("NOT RUN: error_sheet_labeled.csv not found in", OUT)
else:
    el = pd.read_csv(lab_path)
    bad = el[~el["error_category"].isin(CATEGORIES)]
    assert bad.empty, f"invalid/blank error_category in rows {bad.index.tolist()}; allowed: {CATEGORIES}"
    counts = pd.crosstab(el["error_category"], el["system"]).reindex(CATEGORIES, fill_value=0)
    print(counts.to_string())
    examples = {}
    for cat in CATEGORIES:
        if cat == "none": continue
        d = el[el["error_category"] == cat]
        examples[cat] = d.sample(min(2, len(d)), random_state=SEED)[["id", "system", "src", "ref", "hyp", "notes"]].fillna("").to_dict("records")
        for e in examples[cat]: print(f"\n[{cat}] {e['system']} #{e['id']}\n  SRC: {e['src']}\n  REF: {e['ref']}\n  HYP: {e['hyp']}")
    save_results("part4", {"best": best, "worst": worst, "n_labeled": len(el),
                           "counts": {c: counts.loc[c].to_dict() for c in CATEGORIES}, "examples": examples})

system             NLLB-200-600M  opus-mt-en-de
error_category                                 
terminology                    5              4
named entity                   0              0
number/date                    1              0
fluency                        4              5
omission/addition              0              0
mistranslation                 1              2
none                          19             19

[terminology] NLLB-200-600M #70
  SRC: After a week of losses in the midterm election, Bush told an audience about the expansion of trade in Asia.
  REF: Nach einer Woche voller Verluste in der Zwischenwahl erzählte Bush dem Publikum von der Ausweitung des Handels in Asien.
  HYP: Nach einer Woche Verlusten bei den Mittelfristigen Wahlen erzählte Bush einem Publikum von der Expansion des Handels in Asien.

[terminology] opus-mt-en-de #265
  SRC: The bill requires violent video games sold in the state of California to be labeled with a decal reading "18" and mak

## Finish: generate README.md and CV_SNIPPET.md from results.json

In [45]:
# ---- README + CV snippet. Every number is read from results.json; missing -> NOT RUN ----
R = load_results()
def g(*path):
    x = R
    for p in path:
        if not isinstance(x, dict) or p not in x: return None
        x = x[p]
    return x
def num(x, nd=1, suffix=""): return "NOT RUN" if x is None else f"{x:.{nd}f}{suffix}"
def cv(x, nd=1, suffix=""): return "[NOT RUN]" if x is None else f"{x:.{nd}f}{suffix}"

smoke = g("meta", "smoke_test")
banner = ["> **SMOKE TEST OUTPUT - these numbers come from a tiny debug run and are NOT real results.**", ""] if smoke else []

# Part 1 table
systems = g("part1", "systems") or {}
t1 = ["| system | chrF | BLEU | COMET |", "|---|---|---|---|"]
for k, v in systems.items(): t1.append(f"| {k} | {num(v['chrF'])} | {num(v['BLEU'])} | {num(v['COMET'], 3)} |")
if not systems: t1.append("| NOT RUN | NOT RUN | NOT RUN | NOT RUN |")
sig = next(iter(systems.values()), {})
sigs = f"sacreBLEU signatures: `{sig.get('chrF_signature')}`, `{sig.get('BLEU_signature')}`." if sig else "sacreBLEU signatures: NOT RUN."

# Part 2 table
th = g("part2", "thresholds") or {}
desc = {"empty": "empty / whitespace-only source or target",
        "duplicate": "exact duplicate after NFKC + lowercase + whitespace normalisation (first copy kept)",
        "length": f"chars < {th.get('min_chars')} or words > {th.get('max_words')} or length ratio > {th.get('max_len_ratio')}" if th else "NOT RUN",
        "lang_id": f"fastText lid.176: source not en or target not de, or confidence < {th.get('lid_min_conf')}" if th else "NOT RUN",
        "number_mismatch": "digit sets of source and target differ"}
pr = g("part2", "per_rule") or {}
t2 = ["| # | rule | threshold | pairs failing rule | removed (first failing rule) |", "|---|---|---|---|---|"]
for i, (r, d) in enumerate(desc.items(), 1):
    t2.append(f"| {i} | {r} | {d} | {num(pr.get(r, {}).get('failed_any'), 0)} | {num(pr.get(r, {}).get('removed_first'), 0)} |")
p2 = g("part2") or {}
p2line = (f"Removed **{p2['n_removed']} of {p2['n_total']}** pairs ({p2['pct_removed']:.1f}%); kept {p2['n_kept']}. "
          f"Source: `{p2['source']['hf_id']}` ({p2['source']['config']}), {p2['source']['sampling']}, "
          f"from {p2['source']['total_rows']} pairs." if p2 else "Filter: NOT RUN.")

# Part 3
def audit_line(key, label):
    a = g("part3", key)
    return f"- {label}: NOT RUN" if a is None else f"- {label}: {a['bad']}/{a['n']} = {a['pct']:.1f}% (Wilson 95% CI {a['wilson95_low']:.1f}-{a['wilson95_high']:.1f}%)"
audit = [audit_line("removal_precision", "Removal precision (removed pairs that were truly bad)"),
         audit_line("residual_noise_kept", "Residual noise in kept set (kept pairs that were bad)"),
         "- n=100 per group gives wide uncertainty; treat these as rough estimates."]

# Part 4
counts = g("part4", "counts")
if counts:
    names = list(next(iter(counts.values())).keys())
    t4 = ["| category | " + " | ".join(names) + " |", "|---|" + "---|" * len(names)]
    for c, row in counts.items(): t4.append(f"| {c} | " + " | ".join(str(row[n]) for n in names) + " |")
    t4.append(f"\nBest system by COMET: {g('part4','best')}; worst: {g('part4','worst')}. Labeled errors: {g('part4','n_labeled')}.")
else: t4 = ["NOT RUN"]

readme = banner + [
"# MT Quality Lab (EN→DE)", "",
"[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gowrikamahesh17/mt-quality-lab/blob/main/mt_quality_lab.ipynb)", "",
"## Purpose",
"Compare two open MT systems (NLLB-200-distilled-600M, Marian opus-mt-en-de) on FLORES-200 EN→DE with chrF, BLEU and COMET, "
"build and manually audit a rule-based filter for a noisy parallel corpus, and categorise translation errors by hand.", "",
"## Results: FLORES-200 devtest, EN→DE", f"n = {g('part1','n_sentences') or 'NOT RUN'} sentences. COMET = `{CONFIG['comet_model']}`.", "", *t1, "", sigs, "",
"## Corpus filter", p2line, "", *t2, "",
"Pairs can fail several rules; the last column attributes each removed pair to the first rule it fails (rules applied in the order above).", "",
"## Filter audit (manual, 100 removed + 100 kept pairs)", *audit, "",
"## Error analysis (manual, 30 sentences from each of best and worst system)", *t4, "",
"## How to run",
"**Colab:** click the badge, set runtime to T4 GPU, set `SMOKE_TEST = False`, run all. Outputs go to `MyDrive/mt_quality_lab/full/`. "
"Run once to produce `audit_sheet.csv` and `error_sheet.csv`, label them (save as `audit_sheet_labeled.csv` / `error_sheet_labeled.csv` in the same folder), then rerun; finished steps are cached.", "",
"**Local smoke test:** `pip install -r requirements.txt`, keep `SMOKE_TEST = True`, run the notebook top to bottom (outputs in `outputs/smoke/`).", "",
"## Limitations",
"- FLORES-200 may overlap with the training data of the evaluated models (contamination), so scores may be optimistic.",
"- The audit has n=100 per group, which gives wide confidence intervals.",
"- One corpus and domain only (WikiMatrix, Wikipedia-derived); results may not transfer.",
"- Cometinho (a small distilled model) is used instead of the full COMET model; scores are not comparable to full COMET numbers.",
"- Labels come from a single annotator.", "",
"## Data notes",
"FLORES-200 is loaded from Meta's official tarball (the Hugging Face copies are gated or script-based). Model IDs, library versions and sacreBLEU signatures are recorded in `results.json`.", "",
"## What I learned", "TODO: write this section.", ""]

bs = max(systems, key=lambda k: systems[k]["COMET"]) if systems else None
a3 = g("part3"); n_aud = (a3["removal_precision"]["n"] + a3["residual_noise_kept"]["n"]) if a3 else None
cvl = banner + [
f"- Evaluated 2 MT systems (NLLB-200, Marian opus-mt) on FLORES-200 EN-DE with chrF, BLEU and COMET; best system reached chrF {cv(systems[bs]['chrF']) if bs else '[NOT RUN]'} and COMET {cv(systems[bs]['COMET'], 3) if bs else '[NOT RUN]'}",
f"- Built a parallel-corpus filter (deduplication, length ratio, language ID, number mismatch) that removed {cv(g('part2','pct_removed'), 1, '%')} of {g('part2','n_total') or '[NOT RUN]'} sentence pairs, with {cv(g('part3','removal_precision','pct'), 1, '%')} filter precision in a manual check of {n_aud or '[NOT RUN]'} pairs",
f"- Analyzed {g('part4','n_labeled') or '[NOT RUN]'} translation errors by category across the best and worst system", ""]
(OUT / "README.md").write_text("\n".join(readme), encoding="utf-8")
(OUT / "CV_SNIPPET.md").write_text("\n".join(cvl), encoding="utf-8")
print("\n".join(cvl)); print("README.md and CV_SNIPPET.md written to", OUT, "- copy them into the repo root after the full run.")

- Evaluated 2 MT systems (NLLB-200, Marian opus-mt) on FLORES-200 EN-DE with chrF, BLEU and COMET; best system reached chrF 63.7 and COMET 0.553
- Built a parallel-corpus filter (deduplication, length ratio, language ID, number mismatch) that removed 8.3% of 20000 sentence pairs, with 44.0% filter precision in a manual check of 200 pairs
- Analyzed 60 translation errors by category across the best and worst system

README.md and CV_SNIPPET.md written to outputs/full - copy them into the repo root after the full run.
